## STEP 1: image/ を 32×32 に分割して dataset-prepare/ に保存

In [4]:
:dep image = { version = "0.25", features = ["png", "jpeg"] }


In [5]:
use std::fs;
use std::path::Path;
use image::GenericImageView;

fn step1_prepare_dataset(
    src_dir: &str,
    dst_dir: &str,
) -> Result<usize, Box<dyn std::error::Error>> {
    fs::create_dir_all(dst_dir)?;
    let mut count = 0usize;

    let mut entries: Vec<_> = fs::read_dir(src_dir)?
        .filter_map(|e| e.ok())
        .filter(|e| e.path().extension().map(|ex| ex == "png").unwrap_or(false))
        .collect();

    // ファイル名を16進数値としてパースしてソート
    entries.sort_by_key(|e| {
        let stem = e.path().file_stem().unwrap().to_string_lossy().to_string();
        u64::from_str_radix(&stem, 16).unwrap_or(0)
    });

    for entry in &entries {
        let path = entry.path();
        let stem = path.file_stem().unwrap().to_string_lossy().to_string();
        let img = image::open(&path)?;
        let (width, height) = img.dimensions();
        assert_eq!(height, 32, "高さは32pxであること: {}", stem);

        // ★ Bug 2 修正: 全ファイル均一に width/32 で分割（ハードコードなし）
        let n_tiles = width / 32;
        for i in 0..n_tiles {
            let tile = img.crop_imm(i * 32, 0, 32, 32);
            tile.save(format!("{}/{}_{:03}.png", dst_dir, stem, i))?;
            count += 1;
        }
    }
    Ok(count)
}

// 期待値: 7571ファイル × 32タイル = 242272
match step1_prepare_dataset("image", "dataset-prepare") {
    Ok(n)  => println!("STEP1 完了: {} タイル保存（期待値 242272）", n),
    Err(e) => println!("STEP1 エラー: {}", e),
}

STEP1 完了: 242272 タイル保存（期待値 242272）


()